# Meta Platforms (META): Stochastic Momentum Index

Companion to the [Bubble Analytics article](https://www.bubble-analytics.com/blog/Meta-Platforms-META-Stochastic-Momentum-Index-2026-10-07), dated 2026-10-07.

Run all cells from this folder after installing `requirements.txt` and JupyterLab.
The included frozen data covers 2016-10-07 to 2026-10-06.
The measured backtest starts on 2016-10-26 after any indicator warmup.
Data files are read locally; no account, API key or network connection is needed to run the backtest.
`TEMPLATE.ipynb` is the existing general strategy template and is separate from this completed example.

The account starts with $100,000. Fees are 0.05% on entry and 0.05% on exit.
Signals are assessed after the close; market entries execute at the next session's open.
Stops and targets, where present, follow the code below. Daily bars cannot establish every intraday price sequence.
Open positions remain marked to market at the end of the sample; closed-trade statistics exclude them.
Spreads, additional slippage, financing and taxes are not modelled.
The benchmark uses the same sample and price convention as the article.

These are historical, in-sample results, not a promise of future performance.


In [ ]:
import hashlib
from pathlib import Path
input_hashes = {'META-daily.csv.gz': '61161354763ba449b7eb8d25c1e2b4af785083015c4016300c49d664ef7d2eaf'}
for filename, expected_hash in input_hashes.items():
    assert hashlib.sha256((Path('data')/filename).read_bytes()).hexdigest() == expected_hash, 'Input file changed: ' + filename


## Strategy and backtest


In [ ]:
from IPython.display import display
"""Reproduce the META notebook strategy against frozen adjusted daily prices."""
from pathlib import Path
import json, hashlib, warnings
import numpy as np
import pandas as pd
import backtesting
from backtesting import Backtest, Strategy
from pandas.tseries.holiday import (AbstractHolidayCalendar, Holiday, nearest_workday,
    sunday_to_monday, USMartinLutherKingJr, USPresidentsDay, GoodFriday,
    USMemorialDay, USLaborDay, USThanksgivingDay)

ROOT = Path.cwd()
(ROOT/'snapshots').mkdir(exist_ok=True)
(ROOT/'charts').mkdir(exist_ok=True)
raw = pd.read_csv(ROOT/'data/META-daily.csv.gz', parse_dates=['Date']).set_index('Date')
assert raw.index.is_monotonic_increasing and not raw.index.has_duplicates
assert str(raw.index[0].date()) == '2016-10-07'
assert str(raw.index[-1].date()) == '2026-10-06'
assert np.isfinite(raw.to_numpy()).all()
assert (raw[['Open','High','Low','Close']] > 0).all().all()
assert (raw.High >= raw[['Open','Close','Low']].max(axis=1)).all()
assert (raw.Low <= raw[['Open','Close','High']].min(axis=1)).all()
class EquityHolidays(AbstractHolidayCalendar):
    rules = [Holiday('New Year', month=1, day=1, observance=sunday_to_monday),
             USMartinLutherKingJr, USPresidentsDay, GoodFriday, USMemorialDay,
             Holiday('Juneteenth', month=6, day=19, start_date='2022-01-01', observance=nearest_workday),
             Holiday('Independence', month=7, day=4, observance=nearest_workday),
             USLaborDay, USThanksgivingDay,
             Holiday('Christmas', month=12, day=25, observance=nearest_workday)]
holidays = EquityHolidays().holidays(raw.index[0], raw.index[-1]).union(pd.DatetimeIndex(['2018-12-05','2025-01-09']))
expected = pd.bdate_range(raw.index[0], raw.index[-1]).difference(holidays)
assert raw.index.equals(expected.rename('Date')), 'Incomplete trading session sample'

def indicators(frame):
    data = frame[['Open','High','Low','Close']].copy()
    hh = data.High.rolling(10).max()
    ll = data.Low.rolling(10).min()
    displacement = data.Close - (hh + ll) / 2
    numerator = displacement.ewm(span=3, adjust=True).mean().ewm(span=3, adjust=True).mean()
    denominator = (hh-ll).ewm(span=3, adjust=True).mean()
    data['SMI'] = 100 * numerator / denominator
    data['SIG'] = data.SMI.rolling(5).mean()
    data['HISTO'] = (data.SMI-data.SIG).ewm(span=9, adjust=True).mean()
    data = data.dropna().copy()
    data['signal'] = ((data.HISTO < 0) & (data.HISTO > data.HISTO.shift(1))).astype(int)
    data['signal-marker'] = np.where(data.signal.eq(1), data.Low * 0.998, np.nan)
    return data

data = indicators(raw)
for end in (25, 200, 1000, len(raw)-1):
    pd.testing.assert_frame_equal(indicators(raw.iloc[:end]), data.loc[:raw.index[end-1]])
legacy = ((data.HISTO < 0) & (data.HISTO > np.roll(data.HISTO.to_numpy(),1))).astype(int)
assert legacy.iloc[1:].equals(data.signal.iloc[1:])
assert data.signal.iloc[0] == 0

def SIGNAL(): return data.signal
class MetaSMI(Strategy):
    def init(self): self.signal = self.I(SIGNAL)
    def next(self):
        price = float(self.data.Close[-1])
        if self.signal[-1] == 1 and (self.position.is_short or not self.position):
            self.position.close()
            self.buy(size=0.99, sl=0.9*price, tp=1.045*price)

bt = Backtest(data, MetaSMI, cash=100_000, margin=1, exclusive_orders=True,
              commission=0.0005, trade_on_close=False, finalize_trades=False)
with warnings.catch_warnings(record=True) as captured:
    warnings.simplefilter('always')
    stats = bt.run()
trades = stats['_trades'].copy()
equity = stats['_equity_curve'].copy()
benchmark = (100000 * data.Close / data.Close.iloc[0]).rename('BenchmarkEquity')
benchmark_dd = benchmark / benchmark.cummax() - 1
assert backtesting.__version__ == '0.6.5'
assert len(trades) == stats['# Trades']
fees = trades.Size.abs() * (trades.EntryPrice + trades.ExitPrice) * 0.0005
assert np.allclose(trades.Commission, fees)
assert np.allclose(trades.PnL, trades.Size*(trades.ExitPrice-trades.EntryPrice)-fees)
assert np.allclose(trades.ReturnPct, trades.PnL/(trades.Size*trades.EntryPrice))
assert np.isclose((trades.PnL>0).mean()*100,stats['Win Rate [%]'])
assert np.isclose((equity.Equity.iloc[-1]/100000-1)*100,stats['Return [%]'])
assert np.isclose((benchmark.iloc[-1]/100000-1)*100,stats['Buy & Hold Return [%]'])
assert np.isclose((equity.Equity/equity.Equity.cummax()-1).min()*100,stats['Max. Drawdown [%]'])
assert trades.Size.gt(0).all()
exposure = np.zeros(len(data),dtype=bool)
for t in trades.itertuples():
    exposure[t.EntryBar:t.ExitBar+1] = True
for t in stats['_strategy'].trades:
    exposure[t.entry_bar:] = True
assert np.isclose(exposure.mean()*100,stats['Exposure Time [%]'])
for t in trades.itertuples():
    assert data.signal.iloc[t.EntryBar-1] == 1
    assert np.isclose(t.EntryPrice, data.Open.iloc[t.EntryBar])
    assert np.isclose(t.TP, 1.045*data.Close.iloc[t.EntryBar-1])
closed_pnl = float(trades.PnL.sum())
opened = stats['_strategy'].trades
open_fees = sum(t.size*t.entry_price*0.0005 for t in opened)
open_gross = sum(t.size*(data.Close.iloc[-1]-t.entry_price) for t in opened)
assert np.isclose(100000+closed_pnl+open_gross-open_fees,equity.Equity.iloc[-1])
# Rebuild every daily balance independently from dated ledger cash flows and holdings.
reconstructed = pd.Series(100000.0,index=data.index)
for t in trades.itertuples():
    live = (data.index >= t.EntryTime) & (data.index < t.ExitTime)
    reconstructed.loc[live] += t.Size*(data.Close.loc[live]-t.EntryPrice)-t.Size*t.EntryPrice*0.0005
    reconstructed.loc[data.index >= t.ExitTime] += t.PnL
for t in opened:
    live = data.index >= t.entry_time
    reconstructed.loc[live] += t.size*(data.Close.loc[live]-t.entry_price)-t.size*t.entry_price*0.0005
assert np.allclose(reconstructed, equity.Equity)
trades['CumulativePnL'] = trades.PnL.cumsum()
trades['SignalClose'] = [float(data.Close.iloc[int(i)-1]) for i in trades.EntryBar]
trades['InitialStop'] = 0.9*trades.SignalClose
trades['InitialTarget'] = 1.045*trades.SignalClose
trades['ExitKind'] = np.where(trades.ExitPrice <= trades.InitialStop*(1+1e-10), 'Stop', 'Target')
assert ((trades.ExitKind=='Stop') | (trades.ExitPrice>=trades.InitialTarget*(1-1e-10))).all()
metrics = {str(k): str(v) if isinstance(v,(pd.Timestamp,pd.Timedelta)) else float(v)
           for k,v in stats.items() if not str(k).startswith('_') and isinstance(v,(int,float,np.number,pd.Timestamp,pd.Timedelta))}
metrics.update(symbol='META', company='Meta Platforms', initialCash=100000,
    commissionPerSide=0.0005, commissionTotalRate=0.001, dataFirst=str(raw.index[0].date()),
    dataLast=str(raw.index[-1].date()), rawRows=len(raw), measuredRows=len(data),
    warmupRows=len(raw)-len(data), firstStrategyEvaluation=str(data.index[1].date()),
    benchmarkMaxDrawdownPct=float(benchmark_dd.min()*100), benchmarkFinalEquity=float(benchmark.iloc[-1]),
    closedTradePnL=closed_pnl, closedTradeFees=float(fees.sum()), openTradesAtEnd=len(opened),
    openTradeEntryFees=float(open_fees), unrealisedGrossPnLAtEnd=float(open_gross),
    unrealisedPnLAtEnd=float(open_gross-open_fees), winningTrades=int(trades.PnL.gt(0).sum()),
    losingTrades=int(trades.PnL.lt(0).sum()), signalDays=int(data.signal.sum()),
    firstRowSignalChanged=bool(legacy.iloc[0]!=data.signal.iloc[0]),
    targetExits=int(trades.ExitKind.eq('Target').sum()), stopExits=int(trades.ExitKind.eq('Stop').sum()),
    gapAboveTargetSameDayTrades=int(((trades.EntryPrice>trades.InitialTarget)&(trades.EntryTime==trades.ExitTime)).sum()),
    pendingOrdersAtEnd=len(stats['_strategy'].orders),
    meanWinPct=float(trades.loc[trades.PnL>0,'ReturnPct'].mean()*100),
    meanLossPct=float(trades.loc[trades.PnL<0,'ReturnPct'].mean()*100),
    randomSeed=20261007, randomisation='100 permutations of summed net closed-trade percentage returns; no compounding',
    summedTradeReturnPct=float(trades.ReturnPct.sum()*100),
    openTrades=[{'entry':str(t.entry_time),'entryPrice':float(t.entry_price),'size':int(t.size),
                 'stop':float(t.sl),'target':float(t.tp)} for t in opened],
    worstTrade=trades.loc[trades.ReturnPct.idxmin(),['EntryTime','ExitTime','ReturnPct','EntryPrice','ExitPrice','InitialStop']].astype(str).to_dict(),
    backtestingVersion=backtesting.__version__, warnings=[str(w.message) for w in captured],
    snapshotSha256=hashlib.sha256((ROOT/'data/META-daily.csv.gz').read_bytes()).hexdigest(),
    verification={'sessionCoverage':True,'prefixCausality':True,'fees':True,'ledgerReturns':True,
                  'winRate':True,'dailyEquityReconstruction':True,'benchmark':True,'drawdown':True,
                  'exposure':True})
trades.to_csv(ROOT/'snapshots/trades.csv',index=False)
equity.to_csv(ROOT/'snapshots/equity.csv')
data.to_csv(ROOT/'snapshots/indicators.csv')
benchmark.to_csv(ROOT/'snapshots/benchmark.csv')
rng = np.random.default_rng(20261007)
curves = np.array([np.r_[0, np.cumsum(rng.permutation(trades.ReturnPct.to_numpy())*100)] for _ in range(100)])
assert np.allclose(curves[:,-1],metrics['summedTradeReturnPct'])
np.save(ROOT/'snapshots/trade-order-curves.npy',curves)
(ROOT/'snapshots/results.json').write_text(json.dumps(metrics,indent=2,allow_nan=False)+'\n')
display(pd.Series({k: v for k, v in metrics.items() if not isinstance(v, (dict, list))}, name='Result'))


## Charts and account results


In [ ]:
"""Keep the source notebook's Plotly design and regenerate verified values."""
from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROOT = Path.cwd()
(ROOT/'charts').mkdir(exist_ok=True)
m = json.loads((ROOT/'snapshots/results.json').read_text())
d = pd.read_csv(ROOT/'snapshots/indicators.csv',index_col='Date',parse_dates=True)
t = pd.read_csv(ROOT/'snapshots/trades.csv',parse_dates=['EntryTime','ExitTime'])
e = pd.read_csv(ROOT/'snapshots/equity.csv',index_col=0,parse_dates=True)
b = pd.read_csv(ROOT/'snapshots/benchmark.csv',index_col=0,parse_dates=True).iloc[:,0]
curves = np.load(ROOT/'snapshots/trade-order-curves.npy')
CHARTS = {}

def themed(fig,title,yaxis):
    fig.update_layout(title=title,template='plotly_dark',autosize=False,width=1100,height=700,
        yaxis_title=yaxis,font=dict(family='Arial, Helvetica, sans-serif',size=14),
        margin=dict(l=95,r=45,t=85,b=80),legend=dict(orientation='h',y=1.03,x=0))
    fig.update_yaxes(gridcolor='#171717')
    fig.update_xaxes(gridcolor='#171717')
    return fig

window=d.iloc[2200:2550]
f=make_subplots(rows=2,cols=1,shared_xaxes=True,row_heights=[0.68,0.32],vertical_spacing=0.05)
f.add_trace(go.Candlestick(x=window.index,open=window.Open,high=window.High,low=window.Low,close=window.Close,
    increasing_line_color='rgba(19,156,19,0.8)',decreasing_line_color='rgba(175,07,49,0.8)',name='META'),row=1,col=1)
f.add_trace(go.Scatter(x=window.index,y=window['signal-marker'],mode='markers',marker=dict(size=7,color='white'),name='Entry signals'),row=1,col=1)
f.add_trace(go.Scatter(x=window.index,y=window.SMI,line=dict(color='red',width=1),name='SMI'),row=2,col=1)
f.add_trace(go.Scatter(x=window.index,y=window.SIG,line=dict(color='white',width=1),name='Signal average'),row=2,col=1)
f.add_trace(go.Bar(x=window.index,y=window.HISTO,marker_color='gray',name='Smoothed histogram'),row=2,col=1)
f.update_layout(template='plotly_dark',width=1100,height=720,autosize=False,xaxis_rangeslider_visible=False,
    font=dict(family='Arial, Helvetica, sans-serif',size=13),margin=dict(l=75,r=175,t=85,b=65),
    annotations=[dict(text='META · Stochastic Momentum Index Strategy',font=dict(color='white',size=16),xref='paper',yref='paper',x=0.02,y=1.10,showarrow=False),
                 dict(text='SMI',font=dict(color='white',size=12),xref='paper',yref='paper',x=0.02,y=0.30,showarrow=False)],
    legend=dict(x=1.02,y=1,font=dict(size=12)))
f.update_yaxes(gridcolor='#171717');f.update_xaxes(gridcolor='#171717',rangebreaks=[dict(bounds=['sat','mon'])])
f.update_yaxes(title_text='Adjusted price ($)',row=1,col=1)
CHARTS['chart-smi']=f

f=go.Figure()
f.add_trace(go.Scatter(x=e.index,y=(e.Equity/100000-1)*100,name='SMI strategy after fees',line=dict(color='#00df9a',width=2)))
f.add_trace(go.Scatter(x=b.index,y=(b/100000-1)*100,name='META buy and hold',line=dict(color='white',width=1.5)))
CHARTS['chart-equity']=themed(f,'META SMI Strategy · Account Returns','Cumulative return (%)')
f=go.Figure()
f.add_trace(go.Scatter(x=e.index,y=(e.Equity/e.Equity.cummax()-1)*100,name='SMI strategy',line=dict(color='#00df9a',width=2)))
f.add_trace(go.Scatter(x=b.index,y=(b/b.cummax()-1)*100,name='META buy and hold',line=dict(color='white',width=1.5)))
CHARTS['chart-drawdown']=themed(f,'META SMI Strategy · Daily Closing Drawdowns','Decline from previous peak (%)')
f=go.Figure(go.Scatter(x=[d.index[0],*t.ExitTime],y=[0,*t.CumulativePnL],mode='lines',name='Cumulative PnL',line=dict(color='#00df9a',shape='hv')))
CHARTS['chart-pnl']=themed(f,'META SMI Strategy · Closed Trade Profit','Cumulative realised profit ($)')
f=go.Figure()
for curve in curves:
    f.add_trace(go.Scatter(y=curve,mode='lines',opacity=0.6,showlegend=False))
CHARTS['chart-trade-order']=themed(f,'META SMI Strategy · 100 Randomised Trade Orders','Sum of net trade returns (percentage points)')
f.update_xaxes(title_text='Closed trade number')
f.add_annotation(text='Fixed seed · Same 139 closed trades · Summed percentages, without compounding',xref='paper',yref='paper',x=0,y=1.04,showarrow=False,font=dict(size=13,color='#aaaaaa'))

rows=[('Closed trades',str(int(m['# Trades']))),('Exposure time',f"{m['Exposure Time [%]']:.2f}%"),
      ('Win rate',f"{m['Win Rate [%]']:.2f}%"),('Strategy return',f"{m['Return [%]']:.2f}%"),
      ('Buy and hold return',f"{m['Buy & Hold Return [%]']:.2f}%"),('Maximum drawdown',f"{m['Max. Drawdown [%]']:.2f}%"),
      ('Average drawdown',f"{m['Avg. Drawdown [%]']:.2f}%"),('Sharpe ratio',f"{m['Sharpe Ratio']:.2f}"),
      ('Sortino ratio',f"{m['Sortino Ratio']:.2f}"),('Calmar ratio',f"{m['Calmar Ratio']:.2f}"),
      ('Best trade',f"{m['Best Trade [%]']:.2f}%"),('Worst trade',f"{m['Worst Trade [%]']:.2f}%"),
      ('Average trade (geometric)',f"{m['Avg. Trade [%]']:.2f}%"),('Profit factor',f"{m['Profit Factor']:.2f}"),
      ('Expectancy (arithmetic)',f"{m['Expectancy [%]']:.2f}%")]
stripe=['#141A1E' if i%2==0 else '#101519' for i in range(len(rows))]
f=go.Figure(go.Table(columnwidth=[1.6,1],header=dict(values=['',''],height=1,fill_color='#161D21',line_color='#161D21',font=dict(size=1,color='#161D21')),
    cells=dict(values=list(zip(*rows)),align=['left','right'],height=34,font=dict(size=13,color='#E6ECEF',family='Arial, Helvetica, sans-serif'),
    fill_color=[stripe,stripe],line_color='rgba(255,255,255,0.08)',line_width=1)))
f.update_layout(width=500,height=105+34*len(rows)+35,paper_bgcolor='#12181B',plot_bgcolor='#161D21',margin=dict(l=30,r=30,t=105,b=35))
f.add_annotation(text='<b>META · Stochastic Momentum<br>Index Strategy</b>',x=0,y=1.15,xref='paper',yref='paper',xanchor='left',showarrow=False,align='left',font=dict(size=12,color='#E6ECEF',family='Arial, Helvetica, sans-serif'))
f.add_annotation(text="<b><span style='color:#00DF9A'>BUBBLE</span><span style='color:white'>ANALYTICS</span></b>",x=1,y=1.15,xref='paper',yref='paper',xanchor='right',showarrow=False,font=dict(size=12,family='Arial, Helvetica, sans-serif'))
f.add_shape(type='rect',xref='paper',yref='paper',x0=0,y0=0,x1=1,y1=1,line=dict(color='rgba(255,255,255,0.08)',width=1),fillcolor='#161D21',layer='below')
CHARTS['stats-summary']=f

# Consistent branding with a dedicated header above titles and legends.
wordmark = "<b><span style='color:#00DF9A'>BUBBLE</span><span style='color:white'>ANALYTICS</span></b>"
for figure in CHARTS.values():
    figure.layout.annotations = tuple(a for a in figure.layout.annotations if 'BUBBLE' not in (a.text or ''))
    width, height = figure.layout.width or 1100, figure.layout.height or 700
    figure.update_layout(width=width, height=height, margin_t=max(figure.layout.margin.t or 0, 150))
    if figure.layout.title.text:
        figure.update_layout(title_y=1-65/height, title_yanchor='top')
    inner_height = height - figure.layout.margin.t - (figure.layout.margin.b or 80)
    figure.add_annotation(text=wordmark, x=1, y=1+(figure.layout.margin.t-20)/inner_height,
        xref='paper', yref='paper', xanchor='right', yanchor='top', showarrow=False,
        font=dict(size=12, family='Arial, Helvetica, sans-serif'))


In [ ]:
for figure in CHARTS.values():
    display(figure)
